##### Part 3 — Load the raw datasets
Referring to code snippets from Python program: 
[notebooks/02_ingest_raw_from_volume.py].

[1] - Create UNITY CATALOG volume: The path

In [0]:
# SETTING CONSTANTS ACCORDING TO:
# workspace.prj_fintech-transaction-reporting-on-databricks-with-spark_sc/
catalog_name = "workspace"
schema_name = "prj_fintech-transaction-reporting-on-databricks-with-spark_sc"
volume_name = "raw_data"

raw_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"


[2] - Create UNITY CATALOG volume: The pathES to the project`s FILES

In [0]:
transactions_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/transactions.csv"
customers_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/customers.csv"
accounts_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/accounts.csv"


[3] - Read project`s FILES into SPARK DataFrames

NOTE: these have been already verified in PRJ-01 Notebook! Otherwise this would have to happen here.)

In [0]:
# READ FILES INTO DATAFRAMES
transactions_df = spark.read.option("header", True).option("inferSchema", True).csv(transactions_path)
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(customers_path)
accounts_df = spark.read.option("header", True).option("inferSchema", True).csv(accounts_path)



[4] - "Light Inspection": Three aggregations on the 1st DataFrame "TRANSACTIONS"

In [0]:
# Libraries
from pyspark.sql import functions as F

# Light inspection
display(transactions_df.groupBy("transaction_status").count().orderBy(F.desc("count")))
display(transactions_df.groupBy("payment_channel").count().orderBy(F.desc("count")))
display(transactions_df.groupBy("transaction_type").count().orderBy(F.desc("count")))


##### Project Question - Answers
###### [Q1] IDENTIFY: columns that may need standardization from file transactions:
Standardization by transforming into CAPITALS:

- transaction_status
- payment_channel

Standardization by forming proper Date-type value: 
- transaction_date


###### [Q2] IDENTIFY: columns that may contain invalid values from file transactions:
- any amount fields (e.g. NULL or NEGATIVE amounts): amount.
- any primary-key and foreign-key field containing NULL: (1) transaction_id, (2) customer_id, (3) account_id.
- those fields which are crucial to identify useful transaction containing NULL: (1) transaction_type, (2) transaction_status.


###### [Q3] IDENTIFY: join keys you expect to use in the silver stage:
1) From LEFT JOIN condition, having file/DataFrame TRANSACTIONS on leading from the left, keeping its all records: 
- transactions-transaction_id,
- transactions-customer_id, 
- transactions-account_id.

2) The FIRST joining table ("on the right - get records or also get no records): 
- transactions-customer_id = customers-customer_id.

3) The SECOND joining table ("on the right - get records or also get no records): 
- transactions-account_id = accounts-account_id.
